# LocallyConnected2D From Scratch

## Import

In [1]:
import os
import sys
import glob
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score

sys.path.append('../../../')

from src.cnn.utils import load_image, batch_load_images
from src.cnn.layers.locally_connected import LocallyConnected2DLayer
from src.cnn.layers.pooling import MaxPooling2DLayer
from src.cnn.layers.flatten import FlattenLayer
from src.cnn.model import CNNModel, ActivationLayer
from src.shared.layer import Dense

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF version:', tf.__version__)

I0000 00:00:1778836514.865652    3335 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0


## Config

In [2]:
TEST_DIR   = '../../data/intel/seg_test'
MODEL_PATH = 'Layer-2-32-64-5x5-maxpool.h5'

IMG_SIZE    = (150, 150)
NUM_CLASSES = 6
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

## Load Test Data

In [3]:
all_paths, all_labels = [], []
for label_idx, cls in enumerate(CLASS_NAMES):
    paths = sorted(glob.glob(os.path.join(TEST_DIR, cls, '*.jpg')))
    all_paths.extend(paths)
    all_labels.extend([label_idx] * len(paths))

X_test = batch_load_images(all_paths, target_size=IMG_SIZE)
y_true = np.array(all_labels, dtype=int)

print(f'X_test : {X_test.shape}')
print(f'y_true : {y_true.shape}')

X_test : (3000, 150, 150, 3)
y_true : (3000,)


## Load Keras Model (Conv2D)

In [4]:
keras_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
keras_model.summary()

I0000 00:00:1778836850.750091    3335 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "Layer-2-32-64-5x5-maxpool"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 146, 146, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 73, 73, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2s_2 (Conv2D)               │ (None, 69, 69, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73984)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │     9,470,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,524,550 (36.33 MB)

 Trainable params: 9,524,550 (36.33 MB)

 Non-trainable params: 0 (0.00 B)

## Evaluasi Keras

In [5]:
t0 = time.perf_counter()
y_pred_keras = np.argmax(
    keras_model.predict(X_test, batch_size=32, verbose=1),
    axis=1
)
t_keras = time.perf_counter() - t0

f1_keras = f1_score(y_true, y_pred_keras, average='macro')
print(f'\nKeras Macro F1 : {f1_keras:.4f}')
print(f'Keras Time     : {t_keras:.2f}s')

W0000 00:00:1778836907.752350    3335 cpu_allocator_impl.cc:82] Allocation of 810000000 exceeds 10% of free system memory.
W0000 00:00:1778836913.750164    3335 cpu_allocator_impl.cc:82] Allocation of 810000000 exceeds 10% of free system memory.
I0000 00:00:1778836915.573668    4305 service.cc:153] XLA service 0x78883002df40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778836915.573739    4305 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778836915.817555    4305 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778836916.153490    4305 cuda_dnn.cc:461] Loaded cuDNN version 92200


 1/94 ━━━━━━━━━━━━━━━━━━━━ 47:15 30s/step

I0000 00:00:1778836945.482083    4305 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


94/94 ━━━━━━━━━━━━━━━━━━━━ 35s 49ms/step

Keras Macro F1 : 0.7845
Keras Time     : 45.04s


## Build Scratch Model

Konversi Conv2D -> LC2D

In [6]:
def calc_out_hw(H, W, kH, kW, stride):
    return (H - kH) // stride + 1, (W - kW) // stride + 1


scratch_layers = []
current_shape  = (IMG_SIZE[0], IMG_SIZE[1], 3)

for layer in keras_model.layers:
    ltype = type(layer).__name__

    if ltype == 'InputLayer':
        continue

    elif ltype == 'Conv2D':
        conv_w, conv_b = layer.get_weights()
        cfg    = layer.get_config()
        kH, kW, Cin, Cout = conv_w.shape
        stride = cfg['strides'][0]

        H, W, _ = current_shape
        out_H, out_W = calc_out_hw(H, W, kH, kW, stride)
        out_positions = out_H * out_W

        flat_w = conv_w.reshape(kH * kW * Cin, Cout)
        lc_w   = np.tile(flat_w, (out_positions, 1, 1))
        lc_b   = np.tile(conv_b, (out_positions, 1))

        scratch_layers.append(LocallyConnected2DLayer(weights=[lc_w, lc_b], config=cfg))
        print(f'  {layer.name} (Conv2D) -> LocallyConnected2DLayer: '
              f'kernel {conv_w.shape} tiled to {lc_w.shape} | '
              f'{current_shape} -> {(out_H, out_W, Cout)}')

        current_shape = (out_H, out_W, Cout)

    elif ltype == 'MaxPooling2D':
        cfg    = layer.get_config()
        pH, pW = cfg['pool_size']
        stride = cfg['strides'][0]

        H, W, C = current_shape
        out_H, out_W = calc_out_hw(H, W, pH, pW, stride)

        scratch_layers.append(MaxPooling2DLayer(layer))
        print(f'  {layer.name} (MaxPooling2D) -> MaxPooling2DLayer: '
              f'{current_shape} -> {(out_H, out_W, C)}')

        current_shape = (out_H, out_W, C)

    elif ltype == 'Flatten':
        H, W, C = current_shape
        flat_size = H * W * C

        scratch_layers.append(FlattenLayer())
        print(f'  {layer.name} (Flatten) -> FlattenLayer: '
              f'{current_shape} -> ({flat_size},)')

        current_shape = (flat_size,)

    elif ltype == 'Dense':
        w, b = layer.get_weights()
        in_f, out_f = w.shape
        dense = Dense(in_f, out_f)
        dense.set_weights(w, b)
        scratch_layers.append(dense)

        act = layer.get_config().get('activation', 'linear')
        if act != 'linear':
            scratch_layers.append(ActivationLayer(act))

        print(f'  {layer.name} (Dense) -> Dense({in_f}, {out_f}) + ActivationLayer({act})')
        current_shape = (out_f,)

scratch_model = CNNModel.from_layers(scratch_layers)
print(f'\nTransfer bobot selesai! Output shape: {current_shape}')

  conv2s_1 (Conv2D) -> LocallyConnected2DLayer: kernel (5, 5, 3, 32) tiled to (21316, 75, 32) | (150, 150, 3) -> (146, 146, 32)
  maxpool_1 (MaxPooling2D) -> MaxPooling2DLayer: (146, 146, 32) -> (73, 73, 32)
  conv2s_2 (Conv2D) -> LocallyConnected2DLayer: kernel (5, 5, 32, 64) tiled to (4761, 800, 64) | (73, 73, 32) -> (69, 69, 64)
  maxpool_2 (MaxPooling2D) -> MaxPooling2DLayer: (69, 69, 64) -> (34, 34, 64)
  flatten (Flatten) -> FlattenLayer: (34, 34, 64) -> (73984,)
  dense_1 (Dense) -> Dense(73984, 128) + ActivationLayer(relu)
  output (Dense) -> Dense(128, 6) + ActivationLayer(softmax)

Transfer bobot selesai! Output shape: (6,)


In [7]:
# Sanity Check
probe = scratch_model.forward(X_test[0])
assert probe.shape == (NUM_CLASSES,)
assert abs(probe.sum() - 1.0) < 1e-4
print('Sanity check OK, probe shape:', probe.shape, '| sum:', round(float(probe.sum()), 6))

Sanity check OK, probe shape: (6,) | sum: 1.0


## Evaluasi Scratch LC2D

In [8]:
n = len(X_test)
y_pred_scratch = np.zeros(n, dtype=int)
t0 = time.perf_counter()

for i, img in enumerate(X_test):
    y_pred_scratch[i] = scratch_model.predict(img)
    if (i + 1) % 500 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (i + 1) * (n - i - 1)
        print(f'  {i+1}/{n} | {elapsed:.1f}s elapsed | ETA: {eta:.1f}s')

t_scratch = time.perf_counter() - t0
f1_scratch = f1_score(y_true, y_pred_scratch, average='macro')

print(f'\nScratch (LC2D) Macro F1 : {f1_scratch:.4f}')
print(f'Scratch Time            : {t_scratch:.2f}s')

  500/3000 | 731.2s elapsed | ETA: 3656.1s
  1000/3000 | 1386.3s elapsed | ETA: 2772.6s
  1500/3000 | 1968.8s elapsed | ETA: 1968.8s
  2000/3000 | 2742.0s elapsed | ETA: 1371.0s
  2500/3000 | 3537.5s elapsed | ETA: 707.5s
  3000/3000 | 4267.8s elapsed | ETA: 0.0s

Scratch (LC2D) Macro F1 : 0.7838
Scratch Time            : 4267.80s


## Hasil Perbandingan

In [9]:
print(f'>>> Keras Macro F1          : {f1_keras:.4f}')
print(f'>>> Scratch (LC2D) Macro F1 : {f1_scratch:.4f}')
print(f'>>> Selisih                 : {abs(f1_keras - f1_scratch):.6f}')

>>> Keras Macro F1          : 0.7845
>>> Scratch (LC2D) Macro F1 : 0.7838
>>> Selisih                 : 0.000674
